# 🤖 Notebook 3: Traditional Machine Learning Pipeline

**Môn học:** Học Máy (CO3001) — Học kỳ I (2025-2026)  
**Nhóm:** 13

### Mục tiêu
- Train & đánh giá các mô hình ML truyền thống trên Adult Dataset
- So sánh hiệu suất giữa các model và giữa Raw vs PCA features
- Feature importance, learning curve, hyperparameter tuning


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns, os, urllib.request, warnings, joblib, time, copy

from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, learning_curve)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, auc,
                              confusion_matrix, classification_report, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

for d in ['../data','data','../features','features','../models','models',
          '../reports/figures','reports/figures','../reports','reports']:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
DATA_DIR  = '../data'    if os.path.isdir('../data')    else 'data'
FEAT_DIR  = '../features' if os.path.isdir('../features') else 'features'
MOD_DIR   = '../models'  if os.path.isdir('../models')  else 'models'
REP_DIR   = '../reports' if os.path.isdir('../reports') else 'reports'
FIG_DIR   = os.path.join(REP_DIR, 'figures')
print('✅ Libraries loaded!')


## 2. Load Dataset & Preprocessing

In [ ]:
COLUMNS = ['age','workclass','fnlwgt','education','education_num',
           'marital_status','occupation','relationship','race','sex',
           'capital_gain','capital_loss','hours_per_week','native_country','income']
TRAIN_URL  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
TEST_URL   = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
TRAIN_PATH = os.path.join(DATA_DIR, 'adult_train.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'adult_test.csv')

if not os.path.exists(TRAIN_PATH):
    print('Downloading...')
    urllib.request.urlretrieve(TRAIN_URL, TRAIN_PATH)
    urllib.request.urlretrieve(TEST_URL,  TEST_PATH)

df_train = pd.read_csv(TRAIN_PATH, names=COLUMNS, sep=r',\s*', engine='python', na_values='?')
df_test  = pd.read_csv(TEST_PATH,  names=COLUMNS, sep=r',\s*', engine='python', na_values='?', skiprows=1)
df_full  = pd.concat([df_train, df_test], ignore_index=True)
df_full['income'] = df_full['income'].str.replace('.', '', regex=False).str.strip()
df = df_full.drop_duplicates().reset_index(drop=True)

numeric_features     = ['age','fnlwgt','education_num','capital_gain','capital_loss','hours_per_week']
categorical_features = ['workclass','education','marital_status','occupation',
                        'relationship','race','sex','native_country']

X = df.drop(columns=['income'])
y = (df['income'] == '>50K').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')


In [ ]:
# Build & fit preprocessing pipeline
def build_preprocessor(impute_num='median', encoding='onehot', scaling='standard'):
    num_imp = KNNImputer(n_neighbors=5) if impute_num=='knn' else SimpleImputer(strategy=impute_num)
    cat_imp = SimpleImputer(strategy='most_frequent')
    scaler  = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}[scaling]
    encoder = (OneHotEncoder(handle_unknown='ignore', sparse_output=False)
               if encoding=='onehot'
               else OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    return ColumnTransformer([
        ('num', Pipeline([('imp', num_imp), ('sc', scaler)]), numeric_features),
        ('cat', Pipeline([('imp', cat_imp), ('enc', encoder)]), categorical_features),
    ])

preprocessor = build_preprocessor(impute_num='median', encoding='onehot', scaling='standard')
X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)

# Get feature names for importance plots
try:
    ohe_names  = preprocessor.named_transformers_['cat']['enc'].get_feature_names_out(categorical_features)
    feat_names = numeric_features + list(ohe_names)
except:
    feat_names = [f'f{i}' for i in range(X_train_proc.shape[1])]

print(f'Preprocessed: train={X_train_proc.shape}, test={X_test_proc.shape}')
print(f'Total features: {len(feat_names)}')


## 3. Feature Engineering — PCA

In [ ]:
for vr in [0.90, 0.95, 0.99]:
    pca_tmp = PCA(n_components=vr, random_state=RANDOM_STATE)
    n = pca_tmp.fit(X_train_proc).n_components_
    print(f'PCA {vr*100:.0f}%: {X_train_proc.shape[1]} → {n} components')

# Dùng PCA 95%
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca_95.fit_transform(X_train_proc)
X_test_pca  = pca_95.transform(X_test_proc)
print(f'\nDùng PCA 95%: {X_train_proc.shape[1]} → {X_train_pca.shape[1]} features')

# Scree plot
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_proc)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_) * 100
fig, ax  = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar)+1), cumvar, 'b-o', markersize=3)
ax.axhline(95, color='red', linestyle='--', label='95% threshold')
ax.axhline(99, color='orange', linestyle='--', label='99% threshold')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA Scree Plot', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Định Nghĩa Các Mô Hình

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    'SVM (RBF)':           SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                          use_label_encoder=False, eval_metric='logloss',
                                          scale_pos_weight=pos_weight, verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                           class_weight='balanced', verbose=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    'Gaussian NB':         GaussianNB(),
}
print(f'✅ {len(MODELS)} models ready')


## 5. Train & Evaluate

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:,1] if hasattr(model,'predict_proba') else None
    return {
        'accuracy' : accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall'   : recall_score(y_te, y_pred, zero_division=0),
        'f1'       : f1_score(y_te, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_te, y_proba) if y_proba is not None else None,
        'train_time': round(train_time, 2)
    }

# Train trên raw features
print('Training trên Raw Preprocessed features...')
print('-' * 65)
results_raw  = {}
trained_models = {}
for name, model in MODELS.items():
    m = evaluate_model(model, X_train_proc, y_train, X_test_proc, y_test)
    results_raw[name]   = m
    trained_models[name] = model
    print(f'[{name:<22}] acc={m["accuracy"]:.4f}  f1={m["f1"]:.4f}  auc={m["roc_auc"]:.4f}  t={m["train_time"]}s')

results_raw_df = pd.DataFrame(results_raw).T
print('\n✅ Training hoàn tất!')


In [ ]:
# Train trên PCA features
print('Training trên PCA (95%) features...')
print('-' * 65)
results_pca = {}
for name, model in MODELS.items():
    m_pca = copy.deepcopy(model)
    m = evaluate_model(m_pca, X_train_pca, y_train, X_test_pca, y_test)
    results_pca[name] = m
    print(f'[{name:<22}] acc={m["accuracy"]:.4f}  f1={m["f1"]:.4f}  auc={m["roc_auc"]:.4f}')

results_pca_df = pd.DataFrame(results_pca).T


## 6. Visualizations & Evaluation

In [ ]:
# Model comparison chart
results_df = results_raw_df.sort_values('accuracy', ascending=False)

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
fig, axes = plt.subplots(1, 5, figsize=(22, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(results_df)))

for ax, metric in zip(axes, metrics_to_plot):
    sorted_df = results_df[metric].sort_values()
    bars = ax.barh(sorted_df.index, sorted_df.values, color=colors, edgecolor='white', alpha=0.9)
    ax.set_xlabel(metric.upper(), fontsize=10)
    ax.set_title(metric.upper(), fontsize=11, fontweight='bold')
    ax.set_xlim(0.6, 1.0)
    for bar, v in zip(bars, sorted_df.values):
        ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=7)

plt.suptitle('Traditional ML — Performance Comparison (Raw Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Results (Raw Features) ===')
print(results_df[['accuracy','precision','recall','f1','roc_auc','train_time']].round(4).to_string())


In [ ]:
# Confusion Matrix — Top 3 models
top3 = results_df.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, top3):
    y_pred = trained_models[name].predict(X_test_proc)
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={results_df.loc[name,"accuracy"]:.4f}  F1={results_df.loc[name,"f1"]:.4f}',
                 fontsize=11, fontweight='bold')
plt.suptitle('Confusion Matrix — Top 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(10, 8))
palette = plt.cm.Set1(np.linspace(0, 0.85, len(results_df)))
for (name, _), color in zip(results_df.iterrows(), palette):
    model = trained_models[name]
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_proc)[:,1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'--', color='grey')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature Importance — Random Forest
best_name  = results_df.index[0]
best_model = trained_models[best_name]

rf_model = trained_models.get('Random Forest')
if rf_model and hasattr(rf_model, 'feature_importances_'):
    importances = rf_model.feature_importances_
    top_n   = min(20, len(feat_names))
    indices = np.argsort(importances)[::-1][:top_n]
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(top_n), importances[indices], color='steelblue', alpha=0.85)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feat_names[i] for i in indices], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance')
    ax.set_title('Random Forest — Top Feature Importances', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/feature_importance_rf.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Learning Curve — Best model
print(f'Learning Curve cho: {best_name}')
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_proc, y_train, cv=5, n_jobs=-1, scoring='f1',
    train_sizes=np.linspace(0.1, 1.0, 8), random_state=RANDOM_STATE)

tr_m, tr_s = train_scores.mean(1), train_scores.std(1)
va_m, va_s = val_scores.mean(1),   val_scores.std(1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, tr_m, 'b-o', label='Training F1')
ax.fill_between(train_sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color='blue')
ax.plot(train_sizes, va_m, 'r-s', label='Validation F1')
ax.fill_between(train_sizes, va_m-va_s, va_m+va_s, alpha=0.15, color='red')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('F1 Score')
ax.set_title(f'Learning Curve — {best_name}', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Hyperparameter Tuning — Random Forest

In [ ]:
print('GridSearchCV cho Random Forest...')
param_grid = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [None, 10, 20],
    'min_samples_split': [2, 5],
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1
)
grid.fit(X_train_proc, y_train)
tuned_rf = grid.best_estimator_
y_pred_tuned = tuned_rf.predict(X_test_proc)

print(f'Best params : {grid.best_params_}')
print(f'Best CV F1  : {grid.best_score_:.4f}')
print(f'Test F1 (orig)  : {results_raw_df.loc["Random Forest","f1"]:.4f}')
print(f'Test F1 (tuned) : {f1_score(y_test, y_pred_tuned):.4f}')


## 8. Lưu Kết Quả & Best Model

In [ ]:
# Lưu results
results_df.to_csv(os.path.join(REP_DIR, 'model_results.csv'))

# Lưu best model + preprocessor
joblib.dump(best_model, os.path.join(MOD_DIR, f'best_model_{best_name.replace(" ","_")}.pkl'))
joblib.dump(preprocessor, os.path.join(MOD_DIR, 'preprocessor.pkl'))
np.save(os.path.join(FEAT_DIR, 'X_train_preprocessed.npy'), X_train_proc)
np.save(os.path.join(FEAT_DIR, 'X_test_preprocessed.npy'),  X_test_proc)
np.save(os.path.join(FEAT_DIR, 'y_train.npy'), y_train.values)
np.save(os.path.join(FEAT_DIR, 'y_test.npy'),  y_test.values)

print(f'✅ Best model  : {best_name}')
print(f'   Accuracy   : {results_df.loc[best_name,"accuracy"]:.4f}')
print(f'   F1         : {results_df.loc[best_name,"f1"]:.4f}')
print(f'   ROC-AUC    : {results_df.loc[best_name,"roc_auc"]:.4f}')
print('\n✅ Lưu xong! Chạy 04_Deep_Learning (Bonus) tiếp theo.')
